[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ach0000/DualScoreMatching/blob/main/tp/auteurs/denoiser_vs_energy_colab.ipynb)

**Notebook des auteurs** (Guth, Kadkhodaie, Simoncelli, 2025), adapté pour Colab et pour un ordinateur sans GPU par le TP « score matching dual » : les cellules marquées « TP » sont des ajouts ; le reste est inchangé.

Ne lit que les performances enregistrées dans les points de contrôle : CPU ou GPU.

**Sur Colab** : `Exécution > Modifier le type d'exécution > GPU T4`, puis exécuter les cellules dans l'ordre.

In [ ]:
# Installation (Colab) : à exécuter en premier ; en local, ne fait que rendre le code importable
GITHUB_USER = "ach0000"  # propriétaire du fork
REPO = "DualScoreMatching"
BRANCH = "main"
# seuls ces fichiers du dépôt sont téléchargés (motifs de git sparse-checkout)
FICHIERS = ['/*.py',
            '/networks/',
            '/tp/tp_outils.py',
            '/models/finalclean_imagenet64_color_lr0.0005_1Msteps10decays/',
            '/models/finalclean_imagenet64_color_denoiser_psco_lr0.0005_1Msteps10decays/']

import os, sys, subprocess
if "google.colab" in sys.modules:
    root = os.path.join("/content", REPO)
    if not os.path.isdir(os.path.join(root, ".git")):
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--no-checkout", "-b", BRANCH,
                        "https://github.com/%s/%s.git" % (GITHUB_USER, REPO), root], check=True)
    subprocess.run(["git", "-C", root, "sparse-checkout", "set", "--no-cone", *FICHIERS], check=True)
    subprocess.run(["git", "-C", root, "checkout", BRANCH], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'einops', 'PyWavelets'], check=True)
    os.chdir(os.path.join(root, ""))
else:
    racine = os.getcwd()
    while not os.path.exists(os.path.join(racine, "main.py")) and os.path.dirname(racine) != racine:
        racine = os.path.dirname(racine)
    os.chdir(os.path.join(racine, ""))
sys.path.insert(0, os.getcwd())
print("Répertoire courant :", os.getcwd())

In [ ]:
# Compatibilité (ajout du TP) : le code des auteurs a été écrit pour un GPU et une version de PyTorch antérieure
# à 2.6 (torch.load lisait alors par défaut des objets Python, et non seulement des poids).
import functools, sys, torch
sys.argv = sys.argv[:1]   # main.parse_args lit la ligne de commande : celle du noyau Jupyter n'est pas la sienne
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not getattr(torch.load, "_tp", False):
    _torch_load = torch.load
    def _load(*args, **kwargs):
        kwargs.setdefault("weights_only", False)               # points de contrôle des auteurs (dépôt de confiance)
        if not torch.cuda.is_available():
            kwargs.setdefault("map_location", "cpu")           # tenseurs enregistrés sur GPU
        return _torch_load(*args, **kwargs)
    _load._tp = True
    torch.load = _load
import networks.model as _nm
if not torch.cuda.is_available() and not getattr(_nm.Model.__init__, "_tp", False):
    # sans GPU, Model n'enveloppe pas le réseau dans DataParallel, alors que main.py lit model.network.module
    _init = _nm.Model.__init__
    def _init_cpu(self, network, *args, **kwargs):
        _init(self, network, *args, **kwargs)
        self.network = torch.nn.DataParallel(self.network)    # sans GPU, DataParallel appelle le réseau tel quel
    _init_cpu._tp = True
    _nm.Model.__init__ = _init_cpu
os.makedirs("tp/resultats/auteurs", exist_ok=True)   # écritures du notebook (au lieu de outputs/, laissé intact)
print("calcul sur", DEVICE)

In [ ]:
# Statistiques d'ImageNet64 (ajout du TP) : data.load_data les lit avant de construire un réseau ; on les reprend des
# points de contrôle des auteurs (aucun téléchargement).
import sys
sys.path.insert(0, "tp")
import tp_outils
tp_outils.ecrire_infos_auteurs()

# Comparing energy vs denoiser model

In [ ]:

import numpy as np
from matplotlib import pyplot as plt
import torch

from tqdm.auto import tqdm, trange

from main import *

params = {
      'text.usetex': False,  # TP : pas de LaTeX sur Colab
      'font.family': 'sans serif'
}
matplotlib.rcParams.update(params)

## Load two models

In [ ]:
def load_exp(name, step="last", log=True, dataloaders=False):
    """ Load an experiment with a given name. step can be an integer, "best", or "last" (default). """
    exp_dir = Path("models") / name

    with open(exp_dir / "args.json") as f:
        args_dict = json.load(f)
    args_dict["dir"] = "models"  # TP : args.json indique "experiments"

    ctx = TrainingContext(**args_dict, step=step, key_remap=None, seed=None, dataloaders=dataloaders, writer=False)
    if log:
        print(f"{name}: retrieved model at step {ctx.step} and test loss {ctx.test_perf.loss:.2e}")

    # Disable DataParallel (needed for Hessian computation)
    ctx.model.network = ctx.model.network.module

    # Put in eval mode and disable gradients with respect to all parameters.
    ctx.model.eval()
    for p in ctx.model.parameters():
        p.requires_grad = False

    # Normalize energies.
    ctx.network.network.log_normalization_constant = ctx.test_perf.log_normalization_constant

    return ctx

ctxs = {
    "energy": load_exp("finalclean_imagenet64_color_lr0.0005_1Msteps10decays"),
    "denoiser": load_exp("finalclean_imagenet64_color_denoiser_psco_lr0.0005_1Msteps10decays"),
}
default_ctx = ctxs["energy"]
device = default_ctx.device
dataset_info = default_ctx.dataset_info
d = dataset_info.dimension

## Compare denoising performance

In [ ]:
t = default_ctx.test_perf.noise_levels.variance.cpu()
energy_mse = ctxs["energy"].test_perf.denoising_errors.mse.cpu()
denoiser_mse = ctxs["denoiser"].test_perf.denoising_errors.mse.cpu()

plt.figure(figsize=(4, 3), layout="constrained")
plt.plot(t, denoiser_mse, color="tab:blue", marker=".", label="Score network")
plt.plot(t, energy_mse, color="tab:red", linestyle="dashed", marker=".", label="Energy network")
plt.plot(t, t, color="black", label="Identity", zorder=-1)
plt.plot(t, torch.full_like(t, default_ctx.dataset_info.variance), color="gray", label="Data variance", zorder=-1)
plt.xlabel("Noise variance $t$")
plt.xscale("log")
plt.xlim(1e-9, 1e3)
# plt.xticks(10. ** np.arange(-9, 4), fontsize=8)
plt.ylabel("Denoising MSE")
plt.yscale("log")
plt.ylim(5e-10, 2e0)
plt.title("Denoising MSE of energy and score networks")
plt.legend()
plt.savefig("tp/resultats/auteurs/denoising_performance.pdf", transparent=True, bbox_inches="tight", pad_inches=0)
plt.show()

In [64]:
f = lambda x, p, t: print(" & ".join([t] + [f"${y:.{p}f}$" for y in -10 * torch.log10(torch.flip(x, [0])[::2])]) + r" \\")
f(t, 0, "Noise variance")
print(r"\midrule")
f(denoiser_mse, 2, "Score network")
f(energy_mse, 2, "Energy network")

Noise variance & $90$ & $75$ & $60$ & $45$ & $30$ & $15$ & $-0$ & $-15$ & $-30$ \\
\midrule
Score network & $90.20$ & $75.47$ & $60.43$ & $47.19$ & $35.58$ & $25.92$ & $18.84$ & $13.31$ & $-0.11$ \\
Energy network & $90.09$ & $75.17$ & $60.45$ & $47.25$ & $35.67$ & $26.01$ & $18.88$ & $13.48$ & $2.53$ \\
